<a href="https://colab.research.google.com/github/Nagasiv-cyber/hull-white-calibration-pricing/blob/main/hull-white-calibration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install QuantLib

import QuantLib as ql
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from datetime import date

# Global handles for use in other cells
yts = None
a = None
sigma = None

# --- 1. Setup Market Data & Environment ---
def setup_market_data():
    try:
        calculation_date = ql.Date(15, 2, 2024)
        ql.Settings.instance().evaluationDate = calculation_date

        helpers = [
            ql.DepositRateHelper(ql.QuoteHandle(ql.SimpleQuote(0.05)), ql.Period(6, ql.Months), 3, ql.TARGET(), ql.Following, False, ql.Actual360()),
            ql.SwapRateHelper(ql.QuoteHandle(ql.SimpleQuote(0.048)), ql.Period(2, ql.Years), ql.TARGET(), ql.Annual, ql.Unadjusted, ql.Thirty360(ql.Thirty360.BondBasis), ql.Euribor6M()),
            ql.SwapRateHelper(ql.QuoteHandle(ql.SimpleQuote(0.045)), ql.Period(5, ql.Years), ql.TARGET(), ql.Annual, ql.Unadjusted, ql.Thirty360(ql.Thirty360.BondBasis), ql.Euribor6M()),
            ql.SwapRateHelper(ql.QuoteHandle(ql.SimpleQuote(0.042)), ql.Period(10, ql.Years), ql.TARGET(), ql.Annual, ql.Unadjusted, ql.Thirty360(ql.Thirty360.BondBasis), ql.Euribor6M())
        ]
        yield_curve = ql.PiecewiseLinearZero(0, ql.TARGET(), helpers, ql.Actual360())
        yield_curve.enableExtrapolation()
        return ql.YieldTermStructureHandle(yield_curve)
    except Exception as e:
        print(f"Error setting up market data: {e}")
        return None

# --- 2. Hull-White Calibration Engine ---
def calibrate_hull_white(yield_curve_handle, swaption_vols):
    model = ql.HullWhite(yield_curve_handle)
    engine = ql.JamshidianSwaptionEngine(model)
    swaption_helpers = []
    index = ql.Euribor6M(yield_curve_handle)

    for expiry, term, vol in swaption_vols:
        helper = ql.SwaptionHelper(expiry, term, ql.QuoteHandle(ql.SimpleQuote(vol)), index,
                                  ql.Period(1, ql.Years), ql.Thirty360(ql.Thirty360.BondBasis),
                                  ql.Actual360(), yield_curve_handle)
        helper.setPricingEngine(engine)
        swaption_helpers.append(helper)

    optimization_method = ql.LevenbergMarquardt(1e-8, 1e-8, 1e-8)
    model.calibrate(swaption_helpers, optimization_method, ql.EndCriteria(1000, 10, 1e-8, 1e-8, 1e-8))
    return model.params()

# --- 3. Execution ---
def run_pipeline():
    global yts, a, sigma
    yts = setup_market_data()
    if yts is None: return

    vols = [
        (ql.Period(1, ql.Years), ql.Period(5, ql.Years), 0.20),
        (ql.Period(5, ql.Years), ql.Period(5, ql.Years), 0.15)
    ]

    params = calibrate_hull_white(yts, vols)
    a, sigma = params[0], params[1]
    print(f"Calibrated Parameters: Mean Reversion = {a:.4f}, Volatility = {sigma:.4f}")

if __name__ == "__main__":
    run_pipeline()

Calibrated Parameters: Mean Reversion = 0.2623, Volatility = 0.0162


In [ ]:
def simulate_short_rate_paths(yts, a, sigma, n_paths=50, horizon=10, steps=100):
    """
    Simulates and visualizes short-rate paths under the Hull-White model.
    """
    try:
        # Define the Hull-White process and path generator
        process = ql.HullWhiteProcess(yts, a, sigma)
        rng = ql.GaussianRandomSequenceGenerator(ql.UniformRandomSequenceGenerator(steps, ql.UniformRandomGenerator()))
        seq = ql.GaussianPathGenerator(process, horizon, steps, rng, False)

        time_grid = np.linspace(0, horizon, steps + 1)
        paths = np.zeros((n_paths, steps + 1))

        for i in range(n_paths):
            sample_path = seq.next().value()
            paths[i, :] = [sample_path[j] for j in range(len(sample_path))]

        # Visualization
        fig = go.Figure()
        for i in range(min(n_paths, 20)):  # Plot first 20 paths for clarity
            fig.add_trace(go.Scatter(x=time_grid, y=paths[i, :], mode='lines',
                                     line=dict(width=1), opacity=0.6, showlegend=False))

        # Add mean path
        fig.add_trace(go.Scatter(x=time_grid, y=np.mean(paths, axis=0),
                                 mode='lines', name='Mean Path', line=dict(color='black', width=3)))

        fig.update_layout(title=f'Hull-White Short Rate Simulation (a={a:.4f}, σ={sigma:.4f})',
                          xaxis_title='Time (Years)', yaxis_title='Short Rate', hovermode='x')
        fig.show()

    except Exception as e:
        print(f"Simulation Error: {e}")

# Run simulation using parameters from previous calibration
# Note: These variable names 'a' and 'sigma' are based on the execution of the previous cell
if 'yts' in locals() and 'a' in locals():
    simulate_short_rate_paths(yts, a, sigma)
else:
    # Fallback if cell hasn't been run in this session
    yts_temp = setup_market_data()
    vols = [(ql.Period(1, ql.Years), ql.Period(5, ql.Years), 0.20), (ql.Period(5, ql.Years), ql.Period(5, ql.Years), 0.15)]
    params = calibrate_hull_white(yts_temp, vols)
    simulate_short_rate_paths(yts_temp, params[0], params[1])

In [ ]:
def price_bermudan_swaption(yts, a, sigma):
    """
    Prices a Bermudan Swaption using a Tree-based Hull-White model.
    """
    # 1. Define the Underlying Swap
    settlement_date = yts.referenceDate()
    index = ql.Euribor6M(yts)

    # Exercise dates (e.g., annually for 5 years)
    exercise_dates = [settlement_date + ql.Period(i, ql.Years) for i in range(1, 6)]
    bermudan_exercise = ql.BermudanExercise(exercise_dates)

    # Swap details: 5Y starting in 1Y
    start_date = settlement_date + ql.Period(1, ql.Years)
    maturity_date = start_date + ql.Period(5, ql.Years)

    fixed_schedule = ql.Schedule(start_date, maturity_date, ql.Period(1, ql.Years), ql.TARGET(),
                                 ql.ModifiedFollowing, ql.ModifiedFollowing, ql.DateGeneration.Forward, False)
    float_schedule = ql.Schedule(start_date, maturity_date, ql.Period(6, ql.Months), ql.TARGET(),
                                 ql.ModifiedFollowing, ql.ModifiedFollowing, ql.DateGeneration.Forward, False)

    underlying_swap = ql.VanillaSwap(ql.VanillaSwap.Payer, 1000000.0, fixed_schedule, 0.045, ql.Thirty360(ql.Thirty360.BondBasis),
                                     float_schedule, index, 0.0, ql.Actual360())

    # 2. Setup Pricing Engines
    # Engine for the Swaption (Lattice/Tree)
    model = ql.HullWhite(yts, a, sigma)
    swaption_engine = ql.TreeSwaptionEngine(model, 40)

    # Engine for the underlying Swap (Standard discounting)
    swap_engine = ql.DiscountingSwapEngine(yts)
    underlying_swap.setPricingEngine(swap_engine)

    # 3. Price the Swaption
    swaption = ql.Swaption(underlying_swap, bermudan_exercise)
    swaption.setPricingEngine(swaption_engine)

    print(f"Bermudan Swaption NPV: {swaption.NPV():,.2f}")
    print(f"Underlying Swap NPV:  {underlying_swap.NPV():,.2f}")

if 'yts' in locals() and 'a' in locals():
    price_bermudan_swaption(yts, a, sigma)
else:
    print("Error: Market data or calibrated parameters not found in kernel.")

Bermudan Swaption NPV: 18,580.22
Underlying Swap NPV:  -8,237.46


## Risk Analysis: Delta and Vega Calculation
This cell implements the sensitivity analysis for the Bermudan Swaption using a finite difference method.

In [ ]:
def calculate_sensitivities(yts_handle, a_param, sigma_param):
    # Re-setup the pricing components locally for the base case
    def get_swaption_npv(yts_h, a_p, sigma_p):
        index = ql.Euribor6M(yts_h)
        settlement_date = yts_h.referenceDate()
        exercise_dates = [settlement_date + ql.Period(i, ql.Years) for i in range(1, 6)]
        bermudan_exercise = ql.BermudanExercise(exercise_dates)

        start_date = settlement_date + ql.Period(1, ql.Years)
        maturity_date = start_date + ql.Period(5, ql.Years)

        fixed_schedule = ql.Schedule(start_date, maturity_date, ql.Period(1, ql.Years), ql.TARGET(),
                                     ql.ModifiedFollowing, ql.ModifiedFollowing, ql.DateGeneration.Forward, False)
        float_schedule = ql.Schedule(start_date, maturity_date, ql.Period(6, ql.Months), ql.TARGET(),
                                     ql.ModifiedFollowing, ql.ModifiedFollowing, ql.DateGeneration.Forward, False)

        underlying_swap = ql.VanillaSwap(ql.VanillaSwap.Payer, 1000000.0, fixed_schedule, 0.045, ql.Thirty360(ql.Thirty360.BondBasis),
                                         float_schedule, index, 0.0, ql.Actual360())

        model = ql.HullWhite(yts_h, a_p, sigma_p)
        swaption_engine = ql.TreeSwaptionEngine(model, 40)
        swaption = ql.Swaption(underlying_swap, bermudan_exercise)
        swaption.setPricingEngine(swaption_engine)
        return swaption.NPV()

    # 1. Base NPV
    base_npv = get_swaption_npv(yts_handle, a_param, sigma_param)

    # 2. Delta: 1bp (0.0001) parallel shift up in the yield curve
    shift = 0.0001
    shifted_yts = ql.YieldTermStructureHandle(ql.ZeroSpreadedTermStructure(yts_handle, ql.QuoteHandle(ql.SimpleQuote(shift))))
    up_npv = get_swaption_npv(shifted_yts, a_param, sigma_param)
    delta = (up_npv - base_npv) # NPV change per 1bp shift

    # 3. Vega: 1% (0.01) absolute shift in sigma
    vol_shift = 0.01
    vega_npv = get_swaption_npv(yts_handle, a_param, sigma_param + vol_shift)
    vega = (vega_npv - base_npv) # Change in NPV for 1% change in vol

    return base_npv, delta, vega

if 'yts' in locals() and 'a' in locals():
    base_npv, delta, vega = calculate_sensitivities(yts, a, sigma)

    print(f"--- Risk Analysis ---")
    print(f"Base Swaption NPV: {base_npv:,.2f}")
    print(f"Delta (1bp shift): {delta:,.2f}")
    print(f"Vega (1% vol shift): {vega:,.2f}")
else:
    print("Dependencies not found. Please run the calibration cell first.")

--- Risk Analysis ---
Base Swaption NPV: 18,580.22
Delta (1bp shift): 186.81
Vega (1% vol shift): 14,722.07


## Advanced Risk Analysis: Gamma and Theta
This section calculates the second-order price sensitivity (Gamma) and the time decay (Theta).

In [ ]:
def calculate_gamma_theta(yts_handle, a_param, sigma_param):
    def get_swaption_npv(yts_h, eval_date=None):
        if eval_date:
            ql.Settings.instance().evaluationDate = eval_date

        index = ql.Euribor6M(yts_h)
        settlement_date = yts_h.referenceDate()
        exercise_dates = [settlement_date + ql.Period(i, ql.Years) for i in range(1, 6)]
        bermudan_exercise = ql.BermudanExercise(exercise_dates)

        start_date = settlement_date + ql.Period(1, ql.Years)
        maturity_date = start_date + ql.Period(5, ql.Years)

        fixed_schedule = ql.Schedule(start_date, maturity_date, ql.Period(1, ql.Years), ql.TARGET(),
                                     ql.ModifiedFollowing, ql.ModifiedFollowing, ql.DateGeneration.Forward, False)
        float_schedule = ql.Schedule(start_date, maturity_date, ql.Period(6, ql.Months), ql.TARGET(),
                                     ql.ModifiedFollowing, ql.ModifiedFollowing, ql.DateGeneration.Forward, False)

        underlying_swap = ql.VanillaSwap(ql.VanillaSwap.Payer, 1000000.0, fixed_schedule, 0.045, ql.Thirty360(ql.Thirty360.BondBasis),
                                         float_schedule, index, 0.0, ql.Actual360())

        model = ql.HullWhite(yts_h, a_param, sigma_param)
        swaption_engine = ql.TreeSwaptionEngine(model, 40)
        swaption = ql.Swaption(underlying_swap, bermudan_exercise)
        swaption.setPricingEngine(swaption_engine)
        return swaption.NPV()

    # Setup
    base_date = ql.Settings.instance().evaluationDate
    h = 0.0001 # 1bp shift

    # 1. Gamma Calculation (Central Difference)
    # NPV(r + h), NPV(r), NPV(r - h)
    yts_up = ql.YieldTermStructureHandle(ql.ZeroSpreadedTermStructure(yts_handle, ql.QuoteHandle(ql.SimpleQuote(h))))
    yts_down = ql.YieldTermStructureHandle(ql.ZeroSpreadedTermStructure(yts_handle, ql.QuoteHandle(ql.SimpleQuote(-h))))

    npv_base = get_swaption_npv(yts_handle)
    npv_up = get_swaption_npv(yts_up)
    npv_down = get_swaption_npv(yts_down)

    gamma = (npv_up - 2 * npv_base + npv_down) / (h**2) / 1e8 # Normalized for 1bp^2

    # 2. Theta Calculation (1 Day decay)
    tomorrow = ql.TARGET().advance(base_date, ql.Period(1, ql.Days))
    # Note: We keep the same yield curve but move the evaluation clock
    npv_tomorrow = get_swaption_npv(yts_handle, eval_date=tomorrow)
    theta = npv_tomorrow - npv_base

    # Reset evaluation date
    ql.Settings.instance().evaluationDate = base_date

    return gamma, theta

if 'yts' in locals():
    gamma, theta = calculate_gamma_theta(yts, a, sigma)
    print(f"--- Advanced Risk Analysis ---")
    print(f"Gamma (per 1bp^2 shift): {gamma:,.4f}")
    print(f"Theta (1-day decay):     {theta:,.2f}")

--- Advanced Risk Analysis ---
Gamma (per 1bp^2 shift): -0.1532
Theta (1-day decay):     8.48


## Comparative Analysis: Bermudan vs. European Swaption
This section calculates the price of a standard European Swaption to quantify the value of early exercise (the Bermudan Premium).

In [ ]:
def compare_with_european_baseline(yts_handle, a_param, sigma_param):
    # 1. Setup common components
    settlement_date = yts_handle.referenceDate()
    index = ql.Euribor6M(yts_handle)
    start_date = settlement_date + ql.Period(1, ql.Years)
    maturity_date = start_date + ql.Period(5, ql.Years)

    fixed_schedule = ql.Schedule(start_date, maturity_date, ql.Period(1, ql.Years), ql.TARGET(),
                                 ql.ModifiedFollowing, ql.ModifiedFollowing, ql.DateGeneration.Forward, False)
    float_schedule = ql.Schedule(start_date, maturity_date, ql.Period(6, ql.Months), ql.TARGET(),
                                 ql.ModifiedFollowing, ql.ModifiedFollowing, ql.DateGeneration.Forward, False)

    underlying_swap = ql.VanillaSwap(ql.VanillaSwap.Payer, 1000000.0, fixed_schedule, 0.045, ql.Thirty360(ql.Thirty360.BondBasis),
                                     float_schedule, index, 0.0, ql.Actual360())

    # 2. Define Exercises
    # European: Only at the start of the swap
    european_exercise = ql.EuropeanExercise(start_date)
    # Bermudan: Annual opportunities (as defined previously)
    exercise_dates = [settlement_date + ql.Period(i, ql.Years) for i in range(1, 6)]
    bermudan_exercise = ql.BermudanExercise(exercise_dates)

    # 3. Setup Model and Engine
    model = ql.HullWhite(yts_handle, a_param, sigma_param)
    # We use the Tree engine for both to ensure a fair comparison
    tree_engine = ql.TreeSwaptionEngine(model, 40)

    # 4. Price both
    euro_swaption = ql.Swaption(underlying_swap, european_exercise)
    euro_swaption.setPricingEngine(tree_engine)
    euro_npv = euro_swaption.NPV()

    bermudan_swaption = ql.Swaption(underlying_swap, bermudan_exercise)
    bermudan_swaption.setPricingEngine(tree_engine)
    bermu_npv = bermudan_swaption.NPV()

    print(f"--- Optionality Comparison ---")
    print(f"European Swaption NPV: {euro_npv:,.2f}")
    print(f"Bermudan Swaption NPV: {bermu_npv:,.2f}")
    print(f"Bermudan Premium:      {bermu_npv - euro_npv:,.2f}")

if 'yts' in locals():
    compare_with_european_baseline(yts, a, sigma)

--- Optionality Comparison ---
European Swaption NPV: 11,112.17
Bermudan Swaption NPV: 18,580.22
Bermudan Premium:      7,468.05


## Sensitivity Analysis: Bermudan Premium vs. Swap Rate
This analysis explores how the value of early exercise changes as we vary the fixed rate (strike) of the underlying swap.

In [ ]:
def analyze_premium_vs_rate(yts_handle, a_param, sigma_param):
    rates = np.linspace(0.02, 0.07, 11)  # Fixed rates from 2% to 7%
    euro_npvs = []
    bermu_npvs = []

    settlement_date = yts_handle.referenceDate()
    index = ql.Euribor6M(yts_handle)
    start_date = settlement_date + ql.Period(1, ql.Years)
    maturity_date = start_date + ql.Period(5, ql.Years)

    fixed_schedule = ql.Schedule(start_date, maturity_date, ql.Period(1, ql.Years), ql.TARGET(),
                                 ql.ModifiedFollowing, ql.ModifiedFollowing, ql.DateGeneration.Forward, False)
    float_schedule = ql.Schedule(start_date, maturity_date, ql.Period(6, ql.Months), ql.TARGET(),
                                 ql.ModifiedFollowing, ql.ModifiedFollowing, ql.DateGeneration.Forward, False)

    model = ql.HullWhite(yts_handle, a_param, sigma_param)
    tree_engine = ql.TreeSwaptionEngine(model, 40)

    for r in rates:
        underlying_swap = ql.VanillaSwap(ql.VanillaSwap.Payer, 1000000.0, fixed_schedule, r, ql.Thirty360(ql.Thirty360.BondBasis),
                                         float_schedule, index, 0.0, ql.Actual360())

        # European
        euro_opt = ql.Swaption(underlying_swap, ql.EuropeanExercise(start_date))
        euro_opt.setPricingEngine(tree_engine)
        euro_npvs.append(euro_opt.NPV())

        # Bermudan
        exercise_dates = [settlement_date + ql.Period(i, ql.Years) for i in range(1, 6)]
        bermu_opt = ql.Swaption(underlying_swap, ql.BermudanExercise(exercise_dates))
        bermu_opt.setPricingEngine(tree_engine)
        bermu_npvs.append(bermu_opt.NPV())

    # Visualization
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=rates, y=euro_npvs, name='European NPV', mode='lines+markers'))
    fig.add_trace(go.Scatter(x=rates, y=bermu_npvs, name='Bermudan NPV', mode='lines+markers'))
    fig.add_trace(go.Scatter(x=rates, y=np.array(bermu_npvs) - np.array(euro_npvs),
                             name='Bermudan Premium', mode='lines', line=dict(dash='dash')))

    fig.update_layout(title='Swaption NPVs vs. Underlying Swap Rate',
                      xaxis_title='Fixed Rate (Strike)', yaxis_title='NPV',
                      hovermode='x unified')
    fig.show()

if 'yts' in locals():
    analyze_premium_vs_rate(yts, a, sigma)

## 3D Risk Surface: Bermudan Premium vs. Rate & Volatility
This visualization creates a 3D surface plot to show the interaction between moneyness (rate) and market uncertainty (volatility) on the value of early exercise.

In [ ]:
def plot_3d_bermudan_surface(yts_handle, a_param):
    rates = np.linspace(0.03, 0.06, 10)
    vols = np.linspace(0.005, 0.03, 10)
    R, V = np.meshgrid(rates, vols)
    premium_surface = np.zeros(R.shape)

    settlement_date = yts_handle.referenceDate()
    index = ql.Euribor6M(yts_handle)
    start_date = settlement_date + ql.Period(1, ql.Years)
    maturity_date = start_date + ql.Period(5, ql.Years)

    fixed_schedule = ql.Schedule(start_date, maturity_date, ql.Period(1, ql.Years), ql.TARGET(),
                                 ql.ModifiedFollowing, ql.ModifiedFollowing, ql.DateGeneration.Forward, False)
    float_schedule = ql.Schedule(start_date, maturity_date, ql.Period(6, ql.Months), ql.TARGET(),
                                 ql.ModifiedFollowing, ql.ModifiedFollowing, ql.DateGeneration.Forward, False)

    for i in range(len(vols)):
        for j in range(len(rates)):
            r_val = R[i, j]
            v_val = V[i, j]

            model = ql.HullWhite(yts_handle, a_param, v_val)
            tree_engine = ql.TreeSwaptionEngine(model, 30)

            underlying_swap = ql.VanillaSwap(ql.VanillaSwap.Payer, 1000000.0, fixed_schedule, r_val, ql.Thirty360(ql.Thirty360.BondBasis),
                                             float_schedule, index, 0.0, ql.Actual360())

            euro_opt = ql.Swaption(underlying_swap, ql.EuropeanExercise(start_date))
            euro_opt.setPricingEngine(tree_engine)

            exercise_dates = [settlement_date + ql.Period(k, ql.Years) for k in range(1, 6)]
            bermu_opt = ql.Swaption(underlying_swap, ql.BermudanExercise(exercise_dates))
            bermu_opt.setPricingEngine(tree_engine)

            premium_surface[i, j] = bermu_opt.NPV() - euro_opt.NPV()

    fig = go.Figure(data=[go.Surface(z=premium_surface, x=rates, y=vols, colorscale='Viridis')])
    fig.update_layout(title='Bermudan Premium Surface',
                      scene = dict(
                          xaxis_title='Swap Rate (Strike)',
                          yaxis_title='Volatility (Sigma)',
                          zaxis_title='Premium (NPV Diff)'),
                      margin=dict(l=0, r=0, b=0, t=40))
    fig.show()

if 'yts' in locals() and 'a' in locals():
    plot_3d_bermudan_surface(yts, a)